[문제 정의]

- 단순한 최종 성적 비교를 넘어, "어떤 요인이 성적의 상승과 하락을 이끄는가?"를 파악하고자 함.

- 전공별로 성적에 영향을 미치는 주요 생활 습관(수면, 교우 시간 등)이 다를 것이라는 가설 설정.

[데이터셋 선택]

- 학생들의 학업 성취도와 생활 습관 데이터(Student_data.csv)를 기본으로 활용.

- (향후 계획) 필요시 전공별 특성을 보완할 외부 데이터 결합 예정.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/Student_data.csv')

display(df.head())

,Student_ID,Gender,Age,Major,Attendance_Pct,Study_Hours_Per_Day,Previous_CGPA,Sleep_Hours,Social_Hours_Week,Final_CGPA
0,ID00001,Male,20,Engineering,83.9,4.4,2.65,9.1,8,2.78
1,ID00002,Female,24,Business,80.7,4.0,3.58,4.0,4,3.76
2,ID00003,Female,20,Mathematics,91.5,3.9,3.29,6.7,4,3.75
3,ID00004,Female,23,Engineering,73.9,8.8,3.48,4.0,6,3.69
4,ID00005,Male,21,Economics,79.8,2.2,2.66,8.7,6,2.34


In [ ]:
print("결측치 확인:\n", df.isnull().sum())

결측치 확인:
 Student_ID             0
Gender                 0
Age                    0
Major                  0
Attendance_Pct         0
Study_Hours_Per_Day    0
Previous_CGPA          0
Sleep_Hours            0
Social_Hours_Week      0
Final_CGPA             0
dtype: int64


In [ ]:
# 성적 편차 칼럼
df['CGPA_Diff'] = df['Final_CGPA'] - df['Previous_CGPA']

# 편차 기준 계산
def categorize_trend(diff):
    if diff >= 0.2:  
        return '성적 향상'
    elif diff <= -0.2: 
        return '성적 하락'
    else:
        return '성적 유지'

df['Trend'] = df['CGPA_Diff'].apply(categorize_trend)

# 테스트
display(df[['Previous_CGPA', 'Final_CGPA', 'CGPA_Diff', 'Trend']].head(10))

,Previous_CGPA,Final_CGPA,CGPA_Diff,Trend
0,2.65,2.78,0.13,성적 유지
1,3.58,3.76,0.18,성적 유지
2,3.29,3.75,0.46,성적 향상
3,3.48,3.69,0.21,성적 향상
4,2.66,2.34,-0.32,성적 하락
...,...,...,...,...
95,3.37,3.49,0.12,성적 유지
96,3.89,4.00,0.11,성적 유지
97,2.47,3.14,0.67,성적 향상
98,3.28,3.64,0.36,성적 향상


# 데이터 저장하고 싶으면 원본(Student_data.csv)은 그대로 두고, 
# 가공된 데이터(df)를 'Student_data_processed.csv'라는 새 이름으로 저장하기
# df.to_csv('../data/Student_data_processed.csv', index=False)

In [ ]:
import pandas as pd

# 1. 원본 데이터 불러오기
# (파일 경로가 다를 경우 '../data/Student_data.csv' 등으로 수정하세요)
df = pd.read_csv('../data/Student_data.csv')

# ==========================================
# [1단계: 전처리]
# ==========================================
print("=== 1. 누락 데이터(결측치) 확인 ===")
print(df.isnull().sum()) # 모든 칼럼이 0이면 누락된 데이터가 없다는 뜻입니다.

print("\n=== 2. 각 데이터의 타입 확인 ===")
print(df.dtypes) 
print("-" * 50)


# ==========================================
# [2단계: 데이터 다듬기]
# ==========================================
# 1. 칼럼 이름 보기 쉽게 한글로 수정
df.rename(columns={
    'Student_ID': 'ID',
    'Gender': '성별',
    'Age': '나이',
    'Major': '전공',
    'Attendance_Pct': '출석률',
    'Study_Hours_Per_Day': '공부시간',
    'Previous_CGPA': '이전성적',
    'Sleep_Hours': '수면시간',
    'Social_Hours_Week': '사교시간',
    'Final_CGPA': '최종성적'
}, inplace=True)

# 2. 최근 성적이 상위 15%인 사람은 높은 점수임을 표시하는 컬럼 추가
top_15_threshold = df['최종성적'].quantile(0.85) # 상위 15% 컷오프 점수 계산
df['상위15%여부'] = df['최종성적'] >= top_15_threshold

# 3. 지난 성적 대비 편차 계산 및 그룹화 컬럼 추가
df['성적편차'] = df['최종성적'] - df['이전성적']

def categorize_trend(diff):
    if abs(diff) <= 0.3:       # 편차가 0.3 이하이면 '성적 유지'
        return '성적 유지'
    elif diff <= -0.5:         # 편차가 -0.5 이하면 '성적 하락'
        return '성적 하락'
    elif diff >= 0.5:          # 편차가 0.5 이상이면 '성적 향상'
        return '성적 향상'
    else:                      
        return '기타 변동'       # 그 외의 애매한 차이 (0.3 초과 ~ 0.5 미만)

df['성적변동'] = df['성적편차'].apply(categorize_trend)

# ★ 가공된 데이터를 새 CSV 파일로 깔끔하게 저장 (발표용 데이터셋)
df.to_csv('Student_data_processed.csv', index=False, encoding='utf-8-sig')


# ==========================================
# [3단계: 결과 출력하기]
# ==========================================
print(f"\n[기준] 전체 학생 기준 상위 15% 최종 성적 커트라인: {top_15_threshold:.2f}점\n")

# 4. '성적 유지' 그룹에서 꾸준히 성적이 높은(상위 15%) 사람 목록
maintain_high = df[(df['성적변동'] == '성적 유지') & (df['상위15%여부'] == True)][['공부시간', '수면시간', '출석률', '사교시간']]
print(f"=== 4. 성적 유지 & 꾸준히 높은 학생 목록 (총 {len(maintain_high)}명) ===")
display(maintain_high.head())

# 5. '성적 하락' 그룹 목록
decline = df[df['성적변동'] == '성적 하락'][['공부시간', '수면시간', '출석률', '사교시간']]
print(f"\n=== 5. 성적 하락 학생 목록 (총 {len(decline)}명) ===")
display(decline.head())

# 6. '성적 향상' 그룹 목록
improve = df[df['성적변동'] == '성적 향상'][['공부시간', '수면시간', '출석률', '사교시간']]
print(f"\n=== 6. 성적 향상 학생 목록 (총 {len(improve)}명) ===")
display(improve.head())

# 7. 각 전공별(데이터상 6개 전공 존재) 최근 성적 상위 15% 학생 목록
print("\n=== 7. 각 전공 분야별 성적 상위 15% 학생 특징 ===")
majors = df['전공'].unique()

for m in majors:
    m_df = df[df['전공'] == m]
    # 해당 전공 내에서의 상위 15% 기준점 계산
    m_top_15_thresh = m_df['최종성적'].quantile(0.85)
    m_top_15 = m_df[m_df['최종성적'] >= m_top_15_thresh][['공부시간', '수면시간', '출석률', '사교시간']]
    
    print(f"\n[{m}] 전공 상위 15% (커트라인: {m_top_15_thresh:.2f}점, 인원: {len(m_top_15)}명)")
    display(m_top_15.head())

=== 1. 누락 데이터(결측치) 확인 ===
Student_ID             0
Gender                 0
Age                    0
Major                  0
Attendance_Pct         0
Study_Hours_Per_Day    0
Previous_CGPA          0
Sleep_Hours            0
Social_Hours_Week      0
Final_CGPA             0
dtype: int64

=== 2. 각 데이터의 타입 확인 ===
Student_ID                 str
Gender                     str
Age                      int64
Major                      str
Attendance_Pct         float64
Study_Hours_Per_Day    float64
Previous_CGPA          float64
Sleep_Hours            float64
Social_Hours_Week        int64
Final_CGPA             float64
dtype: object
--------------------------------------------------

[기준] 전체 학생 기준 상위 15% 최종 성적 커트라인: 3.87점

=== 4. 성적 유지 & 꾸준히 높은 학생 목록 (총 467명) ===


,공부시간,수면시간,출석률,사교시간
8,4.3,5.0,100.0,12
10,2.2,6.8,60.0,14
44,3.9,6.2,90.1,4
47,4.5,9.1,81.0,5
73,4.6,7.5,91.5,4



=== 5. 성적 하락 학생 목록 (총 35명) ===


,공부시간,수면시간,출석률,사교시간
278,4.1,5.7,52.1,11
344,2.2,9.6,59.4,4
363,2.7,9.0,59.4,9
386,2.3,4.5,58.5,6
425,2.3,8.1,56.1,11



=== 6. 성적 향상 학생 목록 (총 425명) ===


...

,공부시간,수면시간,출석률,사교시간
70,4.7,7.0,81.7,2
120,5.9,4.0,88.9,9
204,8.6,6.9,100.0,10
309,2.9,6.8,88.5,10
358,6.3,7.7,100.0,13



[Psychology] 전공 상위 15% (커트라인: 3.91점, 인원: 135명)


,공부시간,수면시간,출석률,사교시간
44,3.9,6.2,90.1,4
85,3.5,6.4,80.5,8
124,12.5,8.5,79.6,11
232,2.3,7.9,100.0,3
245,2.3,6.4,100.0,9



[Computer Science] 전공 상위 15% (커트라인: 3.90점, 인원: 126명)


,공부시간,수면시간,출석률,사교시간
8,4.3,5.0,100.0,12
47,4.5,9.1,81.0,5
96,1.0,5.1,92.1,8
137,4.6,7.9,81.7,6
185,5.2,7.2,100.0,6
